# Position from the delays, one series at a stated voltage

*Fourth notebook of `relcal`. Draft prepared with Claude Code for A. Villano, 2026-10-30.*

One series, one high voltage, so that nothing in the picture is a mixture of voltages. The events are placed on the face by the delays between the 20 % rise times of the channels (`rise_crossing`, `fraction = 0.2`), as in `positions-from-delays.ipynb`:

| quantity | definition |
|---|---|
| `xdel` | `t_E - (t_C + t_D) / 2 - 1 us` |
| `ydel` | `0.866 (t_D - t_C) + 9 us` |
| `phidel` | `atan2(ydel, xdel)` |

(The empirical offsets `-1` and `+9` are those of Fritts's code. The times are of the raw traces, not filtered.)

**Choosing the series.** Set `SERIES` below. The high voltage and the trigger are read from `series.csv` of the output and printed. The series that the run log (`DataSeriesList.xlsx`, column *Notes*) marks `monitoring He-films` (helium films on the detector, not physics data) are refused: `07221124_2331`, `07221125_0547` and `07221201_0118`. The others of the processed series are marked `low-trigger threshold data`.

Needs the output of `data_analysis/R76/processing/r76-d-idx.dataset.toml` with the `rise_crossing` step. Run with the `darkmatter_cli_env` kernel.

In [ ]:
import csv
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib import colors
from matplotlib.patches import Wedge

from goldpan.dataset import Dataset

SERIES = "07221203_1418"          # -53.5 V; the processed series are listed in series.csv of the output

# series that the run log marks "monitoring He-films" (DataSeriesList.xlsx, column Notes): helium films on the detector, not physics data
HE_FILMS = {"07221124_2331", "07221125_0547", "07221201_0118"}

In [ ]:
_here = Path.cwd().resolve()
DEFINITION = next(p / "data_analysis" / "R76" / "processing" / "r76-d-idx.dataset.toml"
                  for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(DEFINITION).spec.output)
LETTERS = "ABCDEF"

with open(OUT / "series.csv", newline="") as fh:
    meta = {r["series"]: r for r in csv.DictReader(fh)}
if SERIES in HE_FILMS:
    raise SystemExit(f"{SERIES} is marked 'monitoring He-films' in the run log: choose another series")
if SERIES not in meta:
    raise SystemExit(f"{SERIES} is not in {OUT / 'series.csv'}; the series are {sorted(meta)}")
m = meta[SERIES]
print(f"series {SERIES}: high voltage {float(m['hv_v']):.1f} V, trigger {m['trigger']}, {m['dumps_in_output']} dumps")

files = sorted((OUT / "NAI-S2").glob(f"{SERIES}_F*.parquet"))
cols = ["event", "trigger_type"] + [f"Side2p{k}.{q}" for k in LETTERS for q in ("integral", "crossing_time_s")]
tab = pa.concat_tables([pq.read_table(f, columns=cols) for f in files])
event = tab["event"].to_numpy()
trigger = np.asarray(tab["trigger_type"].to_pylist(), dtype=str)
t = {k: tab[f"Side2p{k}.crossing_time_s"].to_numpy() * 1e6 for k in LETTERS}                    # microseconds
integral = np.column_stack([tab[f"Side2p{k}.integral"].to_numpy() for k in LETTERS])
print(f"{len(event):,} events in {len(files)} dumps; trigger types: { {str(k): int((trigger == k).sum()) for k in np.unique(trigger)} }")

## Selection

Real triggers (`Unknown`; the label `Physics` marks the random triggers, see `integral-on-the-face.ipynb`), the sum of the positive offset-subtracted integrals of the six channels above `THRESH`, six times that exist, and all six within `WINDOW` of their median (one pulse in the trace).

In [ ]:
S = integral - np.median(integral, axis=0)
total = np.clip(S, 0, None).sum(axis=1)

THRESH = 2.0                       # as in integral-on-the-face.ipynb
WINDOW = 60.0                      # microseconds
OFFSET_X, OFFSET_Y = -1.0, 9.0     # Fritts's empirical offsets

T = np.column_stack([t[k] for k in LETTERS])
xdel = t["E"] - 0.5 * (t["C"] + t["D"]) + OFFSET_X
ydel = 0.866 * (t["D"] - t["C"]) + OFFSET_Y
phidel = np.degrees(np.arctan2(ydel, xdel))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    one_pulse = (np.abs(T - np.nanmedian(T, axis=1, keepdims=True)) < WINDOW).all(axis=1)
real = trigger == "Unknown"
use = real & (total > THRESH) & np.isfinite(T).all(axis=1) & one_pulse
print(f"{len(event):,} events; real triggers {real.sum():,}; with total > {THRESH}: {(real & (total > THRESH)).sum():,}; "
      f"used (six times, one pulse): {use.sum():,} ({use.sum() / len(event):.1%})")
print(f"{'':28}" + "".join(f"{k:>9}" for k in LETTERS))
print(f"{'median 20 % time (us)':28}" + "".join(f"{np.nanmedian(t[k][use]):>9.1f}" for k in LETTERS))
for name, v in [("xdel", xdel), ("ydel", ydel)]:
    print(f"{name:>6}  1/16/50/84/99 %: " + "  ".join(f"{x:7.2f}" for x in np.percentile(v[use], [1, 16, 50, 84, 99])) + "  (us)")

## The detector in the plane of the delays

The `x` axis is drawn reversed, so the face has the orientation of the first notebook (`C` upper left, `D` lower left, `E` right): a later pulse means an event farther from that channel. The scale is a placeholder, `R_US` microseconds for the radius of the face; the axes are microseconds, not millimetres.

In [ ]:
REGIONS = {   # letter -> (inner radius, outer radius, start angle, end angle) in degrees, radius 1 = the face
    "A": (0.90, 1.00, 0, 360), "B": (0.80, 0.90, 0, 360),
    "C": (0.25, 0.80, 60, 180), "D": (0.25, 0.80, 180, 300), "E": (0.25, 0.80, -60, 60),
    "F": (0.00, 0.25, 0, 360),
}
R_US = 14.0       # PLACEHOLDER: the radius of the face in microseconds of delay
XR, YR = 22, 16

def draw_detector_mirrored(ax, radius=R_US, label=True, lw=0.9):
    for k, (r_in, r_out, a0, a1) in REGIONS.items():
        ax.add_patch(Wedge((0, 0), r_out * radius, 180 - a1, 180 - a0, width=(r_out - r_in) * radius,
                           facecolor="none", edgecolor="k", linewidth=lw, zorder=3))
        if label and k in "CDE":
            a = np.deg2rad((a0 + a1) / 2)
            ax.text(-0.52 * radius * np.cos(a), 0.52 * radius * np.sin(a), k, ha="center", va="center", fontsize=12, color="0.25", zorder=5)
    if label:
        ax.text(0, 0, "F", ha="center", va="center", fontsize=9, color="0.25", zorder=5)
    ax.set_aspect("equal")

title = f"{SERIES}, {float(m['hv_v']):.1f} V, trigger {m['trigger']}"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.25, 1]})
h = ax1.hist2d(xdel[use], ydel[use], bins=[200, 160], range=[[-XR, XR], [-YR, YR]], cmap="viridis", norm=colors.LogNorm(), cmin=1, zorder=2)
draw_detector_mirrored(ax1)
ax1.set_xlim(XR, -XR)
ax1.set_ylim(-YR, YR)
ax1.set_xlabel("xdel (us, reversed: toward C and D on the left)")
ax1.set_ylabel("ydel (us, toward C up)")
ax1.set_title(f"{title}\n{use.sum():,} events")
fig.colorbar(h[3], ax=ax1, label="events per bin")
ax2.hist(phidel[use], bins=np.linspace(-180, 180, 181), histtype="step", color="C0")
ax2.set_xlabel("phidel = atan2(ydel, xdel) (degrees)")
ax2.set_ylabel("events per bin")
ax2.set_title("azimuth of the events")
fig.tight_layout()
plt.show()

## The integral of each channel over the plane of the delays

Mean share of the signal that each channel carries (its positive offset-subtracted integral over `total`), by place in the plane of the delays; places with fewer than 20 events are blank. Each panel has its own colour scale (1st to 99th percentile), since the shares differ little between places.

In [ ]:
frac = np.clip(S, 0, None) / np.where(total > 0, total, np.nan)[:, None]
xs, ys = np.linspace(-XR, XR, 89), np.linspace(-YR, YR, 65)
count, _, _ = np.histogram2d(xdel[use], ydel[use], bins=[xs, ys])

fig, axes = plt.subplots(2, 3, figsize=(16, 8.4))
for ax, i, k in zip(axes.flat, range(6), LETTERS):
    s, _, _ = np.histogram2d(xdel[use], ydel[use], bins=[xs, ys], weights=frac[use, i])
    mean = np.where(count >= 20, s / np.maximum(count, 1), np.nan)
    lo, hi = np.nanpercentile(mean, [1, 99])
    c = ax.pcolormesh(xs, ys, mean.T, cmap="magma", vmin=lo, vmax=hi, zorder=2, alpha=0.9)
    draw_detector_mirrored(ax)
    ax.set_xlim(XR, -XR)
    ax.set_ylim(-YR, YR)
    ax.set_title(f"Side2p{k}: mean share of the signal", fontsize=11)
    fig.colorbar(c, ax=ax, shrink=0.8)
fig.suptitle(title)
fig.tight_layout()
plt.show()

**What this shows, and what is not done.** The events of one series at one voltage placed by the delays, and the share of each channel over that plane. Not done: millimetres (`R_US` is a placeholder), the offsets (from Fritts's code, not derived), filtered times. To compare voltages, change `SERIES` and rerun, or see `positions-by-voltage.ipynb`.